In [ ]:
from IPython.display import display
from IPython.display import Markdown
from PIL import Image,ImageDraw
from openai import OpenAI
import pandas as pd
import textwrap
import base64
import json
import io

In [ ]:
def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

# Include your API key file path here 
with open('gpt_key.txt', 'r') as file:
    GPT_API_KEY = file.read()
client = OpenAI(api_key=GPT_API_KEY)

def gpt_call(model,user_content,n_responses=1,T=1,system_prompt="You are a helpful assistant."):
    flag_completion = True
    cost_consumption = 0

    # Add other models in case you want to generate samples with them
    if model=="gpt-4.1-nano":
        prompt_tokens_cost = 0.1e-6
        completion_tokens_cost = 0.4e-6
    elif model=="gpt-4.1-mini":
        prompt_tokens_cost = 0.4e-6
        completion_tokens_cost = 1.6e-6
    elif model=="gpt-4.1":
        prompt_tokens_cost = 2e-6
        completion_tokens_cost = 8e-6
    elif model=="gpt-5.4":
        prompt_tokens_cost = 2.5e-6
        completion_tokens_cost = 15e-6
    elif model=="gpt-5.-mini":
        prompt_tokens_cost = 0.25e-6
        completion_tokens_cost = 2e-6
    elif model=="gpt-5-nano":
        prompt_tokens_cost = 0.05e-6
        completion_tokens_cost = 0.4e-6
    else:
       raise ValueError("Not a valid model name.")

    while flag_completion:
        try:
            completion = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role":"user","content":user_content}
                ],
                n=n_responses,
                temperature=T
            )
            gpt_response = completion.choices
            cost_consumption += completion.usage.prompt_tokens*prompt_tokens_cost
            cost_consumption += completion.usage.completion_tokens*completion_tokens_cost
            flag_completion = False 
        except Exception as e:
            print("Failed with e: {}".format(e))
    return gpt_response,cost_consumption

def encode_image(image_path,width_size=800):
  img = Image.open(image_path)

  if img.width > width_size:
    ratio = width_size/img.width
    new_size = (width_size, int(img.height * ratio))
    img = img.resize(new_size)
  
  buffered = io.BytesIO()
  img.save(buffered, format="PNG", optimize=True)
  img_b64 = base64.b64encode(buffered.getvalue()).decode("utf-8")

  return img_b64

In [ ]:
# Model's name when calling the gpt_call function
# Available models: gpt-4.1-nano, gpt-4.1-mini, gpt-4.1, gpt-5.4, gpt-5.-mini, gpt-5-nano
gpt_model = "gpt-4.1"
total_cost = 0

In [ ]:
# Prompt used to extract the characteristics: Question, Answer
def create_prompt(lecture_title):
    my_prompt = f"""Generate an exam to evaluate the contents of this lecture ({lecture_title}).
The questions and answers should be fully written in English. 
Minimize the number of questions but maximize the content evaluated in the exam. This means, when possible, design questions that cover more than one slide.

For each question, you should provide:
1. "Answer_info": The set of slides that are necessary to answer the question (maximum 3 slides).
2. "Complement_info": The set of slides that complement the information form the main set of slides (maximum 3 slides).
Any other slide not included in these sets will be considered irrelevant for the question.

Output format:

[
{{"Question":"The question here",
"Answer":"A detailed explanation of the answer"
"Answer_info":[1,2,3],
"Complement_info":[4,5,6]
}}
,
...
]

"""
    return my_prompt

In [ ]:
# This code works for the course "Digital Signal Theory" from Kyushu University used in the experiments
# For a new course, you need to include here the information about the lecture materials included in the course (contentsid)
# the number of pages of each material (pages) and the name of the corresponding class (title)
course_id = 207
course_materials = pd.read_csv("./pid{}_material_info.csv".format(course_id))[["contentsid","pages","title"]]

for n_material in range(course_materials.shape[0]):
    material_id = course_materials["contentsid"][n_material]
    max_pages = course_materials["pages"][n_material]
    title = course_materials["title"][n_material]
    
    print("######################################")
    print("ID: {}".format(material_id))
    print("Title: {}".format(title))

    my_prompt = create_prompt(title)
    user_content =[{"type":"text","text":my_prompt}]
    for i in range(1,max_pages+1):
        # This code works for the course "Digital Signal Theory" from Kyushu University used in the experiments (data delivered upon request)
        # For a new course, you need to include here the lecture materials' slides or the educational documents' chunks in an image format.
        # For only-text cases you need to modify the code
        img = encode_image('./pid{}_materials/material_{}_page_{}.png'.format(course_id,material_id,i),1200)
        new_img = {"type": "image_url", "image_url":{"url": f"data:image/png;base64,{img}"}}
        user_content.append(new_img)

    # 10: Number of generation samples (each generation may find different characteristics)
    # This parameter can be changed to obtain more or less diveersity
    response, cost = gpt_call(gpt_model,user_content,10,1,"You are a university professor.")
    total_cost += cost

    print(f"Total cost: {total_cost}")
    print("--------------------------------------")

    for i in range(10):
        try:
            clean_json = response[i].message.content.strip().replace("```json", "").replace("```", "").strip()
            fixed_json = clean_json.replace('\\', '\\\\')
            crrn_result = json.loads(fixed_json)

            with open('./Agent_resources/Questions/pid{}_{}_QA_MINI_{}_test.jsonl'.format(course_id,material_id,i), 'w', encoding='utf-8') as f:
                for entry in crrn_result:
                    json.dump(entry, f)
                    f.write('\n')
        except:
            # Flags errors in the output format. This will be prevented using an SCHEMA in the following version.
            print(f"Error {i}")

In [ ]:
# Prompt used to extract the characteristics: Representative title, Importance within the lecture, Key topics
def create_2nd_prompt(lecture_title):
    my_prompt = f"""Generate labels for the contents covered by this lecture ({lecture_title}).
All your labels should be fully written in English. 

For each slide in the lecture, you should provide:
1. "Title": A title that concisely summarizes the content of this slide.
2. "Importance": A concise description of the importance of this slide in the context of the whole lecture. This description should answer these questions: What was the purpose of the teacher for creating this slide? Why is this slide necessary for this lecture?
3. "Key_topics": A list of the lecture key topics that are covered by this slide.
4. "Ref_slides": A list of slides directly related to this slide (maximum 3 slides).
Any other slide not included in these sets will be considered irrelevant for the question.

Output format:

[
{{"Slide": 1
"Title":"The title of this slide",
"Importance":"The description of the importance of this slide",
"Key_topics":"listed, key topics, of the lecture, covered by this slide",
"Ref_slides":[2,3,4]
}}
,
...
]

"""
    return my_prompt

In [ ]:
course_materials = pd.read_csv("./pid{}_material_info.csv".format(course_id))[["contentsid","pages","title"]]

for n_material in range(3,4):
    material_id = course_materials["contentsid"][n_material]
    max_pages = course_materials["pages"][n_material]
    title = course_materials["title"][n_material]
    
    print("######################################")
    print("ID: {}".format(material_id))
    print("Title: {}".format(title))

    my_prompt = create_2nd_prompt(title)
    user_content =[{"type":"text","text":my_prompt}]
    for i in range(1,max_pages+1):
        # This code works for the course "Digital Signal Theory" from Kyushu University used in the experiments (data delivered upon request)
        # For a new course, you need to include here the lecture materials' slides or the educational documents' chunks in an image format.
        # For only-text cases you need to modify the code
        img = encode_image('./pid{}_materials/material_{}_page_{}.png'.format(course_id,material_id,i),1200)
        new_img = {"type": "image_url", "image_url":{"url": f"data:image/png;base64,{img}"}}
        user_content.append(new_img)

    # 10: Number of generation samples (each generation may find different characteristics)
    # This parameter can be changed to obtain more or less diversity
    response, cost = gpt_call(gpt_model,user_content,10,1,"You are a university professor.")
    total_cost += cost

    print(f"Total cost: {total_cost}")
    print("--------------------------------------")

    for i in range(10):
        try:
            clean_json = response[i].message.content.strip().replace("```json", "").replace("```", "").strip()
            fixed_json = clean_json.replace('\\', '\\\\')
            crrn_result = json.loads(fixed_json)

            with open('./Agent_resources/Importances/pid{}_{}_IMPORTANCE_{}_test.jsonl'.format(course_id,material_id,i), 'w', encoding='utf-8') as f:
                for entry in crrn_result:
                    json.dump(entry, f)
                    f.write('\n')
        except:
            # Flags errors in the output format. This will be prevented using an SCHEMA in the following version.
            print(f"Error {i}")